# DROID - removing background points whose support was carried away

DROID splits each episode into a **background set** (never moves in world coordinates) and a
dynamic set. Some background points sit on objects the gripper later picks up: the point stays
frozen where the object used to be, so its ground truth is wrong from then on.

**The whole idea, in one line.** Project a track into a camera and compare its depth `z` against the
depth map at that pixel, `d`:

```
gap = d - z
    gap ~ 0    the track is on the surface the camera sees   consistent
    gap < 0    something nearer is in the way                occluded
    gap > 0    the camera sees PAST the track                impossible -> its support is gone
```

A present, unoccluded surface point *is* the surface the camera reports, so a positive gap cannot
happen for correct ground truth. Occlusion pushes the gap negative; only a removed support pushes
it positive. That asymmetry is the entire method.

**The rule**: remove a track if some camera saw it on the surface **on the query frame** and then
saw past it by more than `TAU` on at least `MIN_FRAMES` **consecutive** frames - or, separately, if
its visible/occluded flag flips on more than `FLICKER` of the frames in any camera, which means the
depth there is too unstable for the ground truth to be worth anything.

Both halves earn their place. The points are picked on frame 0, so a track that is already off the
surface *there* was never right to begin with - nothing was carried away, the position was simply
wrong, and checking that one frame is enough to say so. And *consecutive* is what separates a
support that left from a pixel that is merely unreliable: a track straddling a depth edge flickers
over the threshold and back all episode, while a carried-away support opens a gap that stays open.

**What `TAU` really measures.** The gap that opens up is the height of the point above whatever the
camera sees once the object is gone - that is, the object's local thickness, not a property of the
motion. A teapot lid lying on a table gives ~60 mm at the knob and ~20 mm across the dome, so a
`TAU` of 30 mm keeps the knob and silently drops the rest of the same lid. Set `TAU` from the
thinnest object you care about, not from the noise floor. Below ~10 mm the stereo noise starts
coming through, so genuinely thin things - cloth, paper - stay invisible to this. It is a
high-precision, low-recall filter by construction.

In [ ]:
import numpy as np, cv2, matplotlib.pyplot as plt, mediapy
from functools import lru_cache
from pathlib import Path

ROOT = next(p / "tapvidmv_dataset" / "droid" for p in
            [Path("~/tapvidmv_workspace").expanduser(), *Path.cwd().parents]
            if (p / "tapvidmv_dataset" / "droid").is_dir())
EPISODES = sorted(p.name for p in ROOT.iterdir() if (p / "tracks_xyz.npy").exists())
print(len(EPISODES), "episodes in", ROOT)


## 1. Load


In [ ]:
EPISODE = 5      #@param {type:"raw"}

name = EPISODES[EPISODE] if isinstance(EPISODE, int) else EPISODE
d = ROOT / name
tracks = np.load(d / "tracks_xyz.npy")                     # (T, N, 3) world
views = sorted(int(p.name) for p in d.iterdir() if p.name.isdigit())
K   = [np.load(d / str(v) / "intrinsics.npy").astype(np.float32) for v in views]
E   = [np.load(d / str(v) / "extrinsics_w2c.npy").astype(np.float32) for v in views]
JPG = [np.load(d / str(v) / "images_jpeg_bytes.npy", allow_pickle=True) for v in views]
DEP = [np.load(d / str(v) / "depth.npy", mmap_mode="r") for v in views]

T, N = tracks.shape[:2]
BG = np.arange(N) < 300      # the background set: DROID always lays the first 300 tracks down as
                             # the static ones (their world path is exactly 0), the rest are arm

def image(v, f):
    return cv2.cvtColor(cv2.imdecode(np.frombuffer(bytes(JPG[v][f]), np.uint8), 1),
                        cv2.COLOR_BGR2RGB)

def project(v):
    """World tracks into view v: pixel (T, N, 2) and camera-space depth (T, N)."""
    p = np.concatenate([tracks, np.ones((T, N, 1), np.float32)], -1)
    c = np.einsum("tij,tnj->tni", E[v], p)
    z = c[..., 2]
    with np.errstate(invalid="ignore", divide="ignore"):
        xy = c[..., :2] / z[..., None]
    return xy * K[v][None, None, :2] + K[v][None, None, 2:], z

print(f"{name}\n  {T} frames, {N} tracks, {len(views)} views, background {BG.sum()}")

## 2. The measurement

One array holds everything: `gap[view, frame, track]`, in metres, `NaN` where the point is off
screen or the depth map has a hole there. A 5x5 median makes it robust to a bad pixel.


In [ ]:
def compute_gaps():
    """gap[v, f, n] = depth map at the track's pixel  -  the track's own depth."""
    out = np.full((len(views), T, N), np.nan, np.float32)
    dy, dx = np.mgrid[-2:3, -2:3].reshape(2, -1)
    for v in views:
        xy, z = project(v)
        H, W = image(v, 0).shape[:2]
        for f in range(T):
            depth = np.asarray(DEP[v][f], np.float32)
            x, y, zt = xy[f, :, 0], xy[f, :, 1], z[f]
            ok = np.isfinite(x) & np.isfinite(y) & (zt > 0)
            xi = np.round(np.where(ok, x, 0)).astype(int)      # round BEFORE bound-checking
            yi = np.round(np.where(ok, y, 0)).astype(int)
            ok &= (xi >= 2) & (xi < W - 2) & (yi >= 2) & (yi < H - 2)
            if not ok.any():
                continue
            patch = depth[yi[ok, None] + dy, xi[ok, None] + dx]
            patch = np.where(patch > 0, patch, np.nan)
            good = np.isfinite(patch).sum(1) >= 4        # enough valid depth in the window
            surface = np.full(len(patch), np.nan, np.float32)
            if good.any():
                surface[good] = np.nanmedian(patch[good], axis=1)
            out[v, f, ok] = surface - zt[ok]
    return out

gap = compute_gaps()
print("gap", gap.shape, f"  measurable {np.isfinite(gap).mean():.0%} of view-frame-track slots")

## 3. The rule

Two independent reasons to drop a background track, either one is enough.

**Its support left.** `streak[v, n]` is the longest stretch of **consecutive** frames on which camera
`v` sees past track `n`. A camera fires if that stretch reaches `MIN_FRAMES` *and* the same camera
saw the track on the surface on the query frame. Any one camera firing is enough - after the object
leaves, only some viewpoints have a clear line to the space it vacated.

**Its visibility will not settle.** A static point seen by a fixed camera should go in and out of
view a handful of times as the arm sweeps past. One whose visible/occluded flag flips on more than
`FLICKER` of the frames is not tracking anything stable - it sits on a depth edge, or on a surface
the stereo cannot resolve - and its ground truth is not worth trusting either way.

In [ ]:
TAU = 0.015       #@param {type:"number"}
MIN_FRAMES = 30   #@param {type:"integer"}
FLICKER = 0.10    #@param {type:"number"}

def longest_run(b):
    """(V, T, N) bool -> (V, N): longest run of consecutive True along the frame axis."""
    best = np.zeros((b.shape[0], b.shape[2]), int)
    run = np.zeros_like(best)
    for f in range(b.shape[1]):
        run = np.where(b[:, f], run + 1, 0)      # NaN gaps compare False, so they break the run
        best = np.maximum(best, run)
    return best

streak  = longest_run(gap > TAU)                 # (V, N) longest stretch seeing PAST the track
onquery = np.abs(gap[:, 0]) <= TAU               # (V, N) was on the surface where it was picked
fires   = (streak >= MIN_FRAMES) & onquery       # its support left

vis     = np.isfinite(gap) & (gap >= -TAU)       # (V, T, N) this camera has a clear line to it
churn   = (vis[:, 1:] != vis[:, :-1]).sum(1) / (T - 1)   # (V, N) how often that flag flips
jitters = churn > FLICKER                        # its visibility never settles

remove  = BG & (fires.any(axis=0) | jitters.any(axis=0))          # the verdict
trigger = np.where(fires.any(axis=0), fires.argmax(axis=0), streak.argmax(axis=0))

print(f"drop a background track if some camera saw it on the surface on frame 0 then saw past it"
      f" by >{TAU*1000:.0f}mm on >={MIN_FRAMES} consecutive frames of {T},")
print(f"or if its visible/occluded flag flips on >{FLICKER:.0%} of frames in any camera\n")
print(f"  removed {remove.sum()} of {BG.sum()} background tracks"
      f"  ({remove.sum()/N:.1%} of all {N})")
print(f"    support left:       {(BG & fires.any(axis=0)).sum():3d}")
print(f"    visibility jitters: {(BG & jitters.any(axis=0)).sum():3d}"
      f"  ({(BG & jitters.any(axis=0) & ~fires.any(axis=0)).sum()} of them only for this reason)")
never = BG & (streak >= MIN_FRAMES).any(axis=0) & ~fires.any(axis=0)
print(f"  spared by the query-frame test: {never.sum():3d}"
      f"  (off the surface already on frame 0, nothing moved)")
for v in views:
    f_v = BG & fires[v]
    alone = f_v & (np.delete(fires, v, axis=0).sum(axis=0) == 0)
    print(f"  view {v}: {f_v.sum():3d} fire, {alone.sum():3d} of them alone")
print(f"\n  removed ids: {np.flatnonzero(remove).tolist()}")

## 4. Look at what it removed

**The curve is the decision.** The camera that fired is drawn solid, the others dashed. The strip
underneath is that same camera - the one that actually saw something - so the picture and the
verdict finally refer to the same thing.


In [ ]:
SHOW = 6   # how many removed tracks to inspect
STRIP = np.unique(np.linspace(0, T - 1, 20).astype(int))   # frames drawn in the image strip

def evidence(track):
    v = int(trigger[track])
    xy, _ = project(v)
    fig, (a, b) = plt.subplots(2, 1, figsize=(13, 4.4),
                               gridspec_kw=dict(height_ratios=[1, 1.1]))
    for w in views:
        a.plot(gap[w][:, track] * 1000, lw=2.2 if w == v else 1.0,
               ls="-" if w == v else "--",
               label=f"view {w}" + ("  <- fired" if w == v else ""))
    a.axhline(TAU * 1000, color="crimson", ls=":", lw=1)
    a.axhline(0, color="0.7", lw=0.8)
    a.set(ylabel="gap (mm)", xlabel="frame",
          title=f"track {track}   view {v}: longest run past the surface ="
                f" {int(streak[v, track])} frames")
    a.legend(fontsize=8, ncol=len(views)); a.grid(alpha=0.3)

    strip = []
    for f in STRIP:
        x, y = xy[f, track]
        if not np.isfinite([x, y]).all():
            strip.append(np.full((90, 90, 3), 70, np.uint8)); continue
        H, W = image(v, 0).shape[:2]
        xi, yi = int(round(float(x))), int(round(float(y)))
        crop = image(v, f)[max(0, yi-30):yi+30, max(0, xi-30):xi+30]
        crop = (cv2.resize(crop, (90, 90), interpolation=cv2.INTER_NEAREST)
                if crop.size else np.full((90, 90, 3), 70, np.uint8))
        cv2.drawMarker(crop, (45, 45), (255, 255, 0), cv2.MARKER_CROSS, 20, 1, cv2.LINE_AA)
        red = gap[v, f, track] > TAU
        cv2.rectangle(crop, (0, 0), (89, 89), (235, 60, 60) if red else (60, 220, 60), 3)
        strip.append(crop)
    b.imshow(np.hstack(strip)); b.axis("off")
    b.set_title(f"view {v}, {len(STRIP)} frames across the episode"
                f"   (red border = this frame is over tau)", fontsize=9)
    fig.tight_layout()

for t in np.flatnonzero(remove)[np.argsort(-streak.max(axis=0)[remove])][:SHOW]:
    evidence(int(t))
plt.show()

## 5. Before and after


In [ ]:
FRAME = 0     # the query frame: where the points were picked, all still on their support

PROJ  = [project(v) for v in views]                  # draw() is called ~300 times by the video
SHAPE = [image(v, 0).shape[:2] for v in views]       # below; neither of these varies with f

@lru_cache(maxsize=8)
def canvas(v, f):
    H, W = SHAPE[v]
    return cv2.resize(image(v, f), (520, int(520 * H / W)))

def draw(v, f, ids, mark=()):
    """Colour is the verdict (red = removed), shape is whether THIS camera can see the point:
    solid = clear line to it, ring = occluded, faint = no depth reading there."""
    img = canvas(v, f).copy()
    s = 520 / SHAPE[v][1]
    xy, z = PROJ[v]
    for n in ids:
        x, y = xy[f, n] * s
        if not (np.isfinite([x, y]).all() and z[f, n] > 0):
            continue
        col = (235, 60, 60) if n in mark else (60, 200, 255)
        g = gap[v, f, n]
        if not np.isfinite(g):
            cv2.circle(img, (int(x), int(y)), 3, tuple(c // 3 for c in col), -1, cv2.LINE_AA)
        elif g < -TAU:
            cv2.circle(img, (int(x), int(y)), 5, col, 1, cv2.LINE_AA)
        else:
            cv2.circle(img, (int(x), int(y)), 4, col, -1, cv2.LINE_AA)
    return img

bg_ids = np.flatnonzero(BG)
rm_ids = set(np.flatnonzero(remove).tolist())
fig, axes = plt.subplots(len(views), 2, figsize=(13, 3.1 * len(views)))
for v in views:
    axes[v, 0].imshow(draw(v, FRAME, bg_ids, rm_ids))
    axes[v, 0].set_title(f"view {v}  BEFORE  ({len(bg_ids)} background, {len(rm_ids)} in red)"
                         f"   solid = visible, ring = occluded, faint = no depth", fontsize=9)
    axes[v, 1].imshow(draw(v, FRAME, [n for n in bg_ids if n not in rm_ids]))
    axes[v, 1].set_title(f"view {v}  AFTER  ({len(bg_ids) - len(rm_ids)} kept)", fontsize=9)
    for ax in axes[v]:
        ax.axis("off")
fig.tight_layout()
plt.show()


In [7]:
# The same thing as a video, one row per camera.
clip = []
for f in np.linspace(0, T - 1, 50).astype(int):
    rows = [np.hstack([draw(v, f, bg_ids, rm_ids),
                       draw(v, f, [n for n in bg_ids if n not in rm_ids])]) for v in views]
    clip.append(np.vstack(rows))
mediapy.show_video(np.stack(clip), fps=10, title=f"{name}   left BEFORE / right AFTER")


AUTOLab+5d05c5aa+2023-07-23-21h-12m-36s left BEFORE / right AFTER This browser does not support the video tag.


## 6. Save the mask


In [ ]:
out = Path("droid_cleaned"); out.mkdir(exist_ok=True)
np.savez(out / f"{name.replace('+', '_')}_keep.npz",
         keep=~remove, removed=np.flatnonzero(remove), streak=streak,
         background=BG, tau=TAU, min_frames=MIN_FRAMES)
print("kept", int((~remove).sum()), "of", N, "->", out.resolve())